#### Lab 0 - 0.1 - CNN Experiments on CIFAR-10

##### Baseline: SGD (lr=0.0001) | LeakyReLU
##### Experiment 1: Adam | LeakyReLU
##### Experiment 2: Adam | Tanh

In [1]:
%%time

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import numpy as np

# 25% GPU
torch.cuda.set_per_process_memory_fraction(0.25, device=0)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

I0000 00:00:1774649586.510063  116137 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774649586.597561  116137 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774649588.593457  116137 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Using device: cuda
CPU times: user 10.5 s, sys: 978 ms, total: 11.5 s
Wall time: 6.51 s


In [2]:
%%time

# Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download and load training set
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True, num_workers=2
)

# Download and load test set
testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False, num_workers=2
)

# CIFAR-10 class labels
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print('Training samples:', len(trainset))
print('Test samples:', len(testset))

/usr/local/lib/python3.11/dist-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Training samples: 50000
Test samples: 10000
CPU times: user 1.61 s, sys: 341 ms, total: 1.95 s
Wall time: 1.95 s


In [5]:
%%time

class CNN(nn.Module):
    def __init__(self, activation_fn=nn.LeakyReLU()):
        super(CNN, self).__init__()
        self.activation_fn = activation_fn

        # Convolutional layers
        self.conv_block = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),                          

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2),                          

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            self.activation_fn,
            nn.MaxPool2d(2, 2)                           
        )

        # Fully connected layers
        self.fc_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            self.activation_fn,
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_block(x)
        x = self.fc_block(x)
        return x


def train_model(model, optimizer, num_epochs, run_name):
    """Training function."""
    criterion = nn.CrossEntropyLoss()
    writer = SummaryWriter(f'runs/{run_name}')

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for i, (inputs, labels) in enumerate(trainloader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(trainloader)
        epoch_acc = 100. * correct / total

        # Tensorboard logging
        writer.add_scalar('Loss/train', epoch_loss, epoch)
        writer.add_scalar('Accuracy/train', epoch_acc, epoch)

        print(f'Epoch [{epoch+1}/{num_epochs}] | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%')

    writer.close()
    print('Training complete!')
    return model


def evaluate_model(model, label):
    """Evaluation function."""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    accuracy = 100. * correct / total
    print(f'Test Accuracy ({label}): {accuracy:.2f}%')
    return accuracy


# Final summary
results = {}
NUM_EPOCHS = 10

CPU times: user 119 μs, sys: 13 μs, total: 132 μs
Wall time: 139 μs


#### Baseline: SGD + LeakyReLU

In [7]:
%%time

# Instantiate model with LeakyReLU
model_exp0 = CNN(activation_fn=nn.LeakyReLU()).to(device)

# SGD optimizer with lr=0.0001
optimizer_exp0 = optim.SGD(model_baseline.parameters(), lr=0.0001)

print('Optimizer: SGD | LR: 0.0001 | Activation: LeakyReLU')

model_exp0 = train_model(model_baseline, optimizer_baseline, NUM_EPOCHS, 'exp0_sgd_leakyrelu')

Optimizer: SGD | LR: 0.0001 | Activation: LeakyReLU
Epoch [1/10] | Loss: 2.2975 | Train Acc: 15.78%
Epoch [2/10] | Loss: 2.2970 | Train Acc: 15.95%
Epoch [3/10] | Loss: 2.2965 | Train Acc: 15.98%
Epoch [4/10] | Loss: 2.2959 | Train Acc: 16.03%
Epoch [5/10] | Loss: 2.2954 | Train Acc: 16.07%
Epoch [6/10] | Loss: 2.2948 | Train Acc: 16.08%
Epoch [7/10] | Loss: 2.2942 | Train Acc: 16.12%
Epoch [8/10] | Loss: 2.2936 | Train Acc: 16.19%
Epoch [9/10] | Loss: 2.2930 | Train Acc: 16.24%
Epoch [10/10] | Loss: 2.2924 | Train Acc: 16.29%
Training complete!
CPU times: user 47.4 s, sys: 6.89 s, total: 54.3 s
Wall time: 1min 8s


In [8]:
%%time

results['Baseline (SGD + LeakyReLU)'] = evaluate_model(model_baseline, 'Baseline - SGD + LeakyReLU')

Test Accuracy (Baseline - SGD + LeakyReLU): 16.31%
CPU times: user 470 ms, sys: 292 ms, total: 762 ms
Wall time: 1.53 s


#### Experiment 1: Adam + LeakyReLU

In [9]:
%%time

# Instantiate model with LeakyReLU
model_exp1 = CNN(activation_fn=nn.LeakyReLU()).to(device)

# Adam optimizer (default lr=0.001)
optimizer_exp1 = optim.Adam(model_exp1.parameters())

print('Optimizer: Adam | Activation: LeakyReLU')

model_exp1 = train_model(model_exp1, optimizer_exp1, NUM_EPOCHS, 'exp1_adam_leakyrelu')

Optimizer: Adam | Activation: LeakyReLU
Epoch [1/10] | Loss: 1.3632 | Train Acc: 50.66%
Epoch [2/10] | Loss: 0.9076 | Train Acc: 68.02%
Epoch [3/10] | Loss: 0.7106 | Train Acc: 75.09%
Epoch [4/10] | Loss: 0.5700 | Train Acc: 80.04%
Epoch [5/10] | Loss: 0.4468 | Train Acc: 84.14%
Epoch [6/10] | Loss: 0.3402 | Train Acc: 88.04%
Epoch [7/10] | Loss: 0.2480 | Train Acc: 91.22%
Epoch [8/10] | Loss: 0.1761 | Train Acc: 93.87%
Epoch [9/10] | Loss: 0.1289 | Train Acc: 95.47%
Epoch [10/10] | Loss: 0.0999 | Train Acc: 96.60%
Training complete!
CPU times: user 50.2 s, sys: 6.53 s, total: 56.7 s
Wall time: 1min 8s


In [10]:
%%time

results['Experiment 1 (Adam + LeakyReLU)'] = evaluate_model(model_exp1, 'Experiment 1 - Adam + LeakyReLU')

Test Accuracy (Experiment 1 - Adam + LeakyReLU): 75.19%
CPU times: user 432 ms, sys: 179 ms, total: 610 ms
Wall time: 1.45 s


#### Experiment 2: Adam + Tanh

In [11]:
%%time

# Instantiate model with Tanh
model_exp2 = CNN(activation_fn=nn.Tanh()).to(device)

# Adam optimizer (default lr=0.001)
optimizer_exp2 = optim.Adam(model_exp2.parameters())

print('Optimizer: Adam | Activation: Tanh')

model_exp2 = train_model(model_exp2, optimizer_exp2, NUM_EPOCHS, 'exp2_adam_tanh')

Optimizer: Adam | Activation: Tanh
Epoch [1/10] | Loss: 1.2448 | Train Acc: 55.62%
Epoch [2/10] | Loss: 0.8658 | Train Acc: 69.88%
Epoch [3/10] | Loss: 0.6986 | Train Acc: 75.87%
Epoch [4/10] | Loss: 0.5639 | Train Acc: 80.48%
Epoch [5/10] | Loss: 0.4547 | Train Acc: 84.34%
Epoch [6/10] | Loss: 0.3554 | Train Acc: 87.82%
Epoch [7/10] | Loss: 0.2674 | Train Acc: 91.20%
Epoch [8/10] | Loss: 0.1974 | Train Acc: 93.71%
Epoch [9/10] | Loss: 0.1536 | Train Acc: 95.16%
Epoch [10/10] | Loss: 0.1219 | Train Acc: 96.10%
Training complete!
CPU times: user 49.9 s, sys: 6.63 s, total: 56.5 s
Wall time: 1min 7s


In [12]:
%%time

results['Experiment 2 (Adam + Tanh)'] = evaluate_model(model_exp2, 'Experiment 2 - Adam + Tanh')

Test Accuracy (Experiment 2 - Adam + Tanh): 73.11%
CPU times: user 442 ms, sys: 161 ms, total: 603 ms
Wall time: 1.5 s


#### Results

In [13]:
print(f'{"Experiment":<35} {"Test Accuracy":>15}')
for name, acc in results.items():
    print(f'{name:<35} {acc:>14.2f}%')

Experiment                            Test Accuracy
Baseline (SGD + LeakyReLU)                   16.31%
Experiment 1 (Adam + LeakyReLU)              75.19%
Experiment 2 (Adam + Tanh)                   73.11%
